# Fine-Tuning v4 — Toti Cakery Tool-Calling (LoRA, Qwen3-1.7B)

Melatih LoRA pada `unsloth/Qwen3-1.7B` (padanan `qwen3:1.7b` di Ollama —
pemenang bake-off v3), lalu
mengukur **baseline (sebelum) vs fine-tuned (sesudah)** pada split `test`
dengan kondisi identik. Export akhir: GGUF **Q4_K_M** (dari sumber f16,
±1GB → lebih cepat di CPU; verdict tetap harness lokal setelah deploy).

**v4 (Jul 2026, PROMPT_FINETUNE_V4.md):** dataset menutup 4 insiden live WA —
history berupa penanda `_history_view` produksi, reminder menempel di system
block (paritas collate Ollama), argumen tool VERBATIM kata pelanggan, dan
**masking loss hanya pada respons assistant FINAL** (akar insiden halusinasi
menu: turn assistant di history ikut terlatih lalu dihafal). Eval in-notebook
merakit prompt persis runtime. **Metrik baseline (sebelum) vs fine-tuned
(sesudah) dihitung pada kondisi identik** — tabel per metrik & per tipe ada di
seksi Compare (untuk tugas akhir metode kuantitatif), diarsipkan juga ke
`hasil_metrik_v4.json`.

Notebook ini ditulis dari template resmi Unsloth + `finetune/INSTRUKSI_FINETUNING.md`.
Jalankan di Colab **T4 GPU**. Estimasi total ±1–1.5 jam. Urutan: training +
tabel perbandingan dulu, export GGUF paling akhir. **Verdict resmi tetap
harness lokal** (`finetune/eval_tool_calling.py` + `scenario_suite.py` R1-R6)
setelah GGUF di-deploy.

### Installation

(Cell instalasi diambil apa adanya dari template resmi Unsloth.)

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

<a name="Data"></a>
### Data Prep — dataset Toti Cakery tool-calling

Dataset: **`LasagnaS/toti-cakery-toolcall`** (public, identik dengan
`finetune/data/` repo). Dimuat MENTAH (kolom `messages` + `tools_json`) —
render ke kolom `text` dilakukan di pipeline dengan chat template Qwen3
(tool call JSON `{"name": ..., "arguments": {...}}`).

⚠️ Split `test` hanya untuk evaluasi — tidak pernah masuk training/val.

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# CATATAN: load_dataset("LasagnaS/toti-cakery-toolcall") langsung bisa gagal di
# datasets versi lama — "Feature type 'Json' not found" — karena metadata yang
# diekspor Hub dibuat dengan datasets versi baru. Unduh jsonl-nya langsung lalu
# muat via builder "json" (tahan semua versi):
files = {split: hf_hub_download("LasagnaS/toti-cakery-toolcall",
                                f"data/{split}.jsonl", repo_type = "dataset")
         for split in ("train", "validation", "test")}
ds = load_dataset("json", data_files = files)

print(ds)
# v4 (Jul 2026): train 1005 / val 102 (§3: penanda history, args verbatim,
# reminder di system, T3/T5 diperbanyak, N5 dipangkas); test 100 dibekukan dari v1
assert len(ds["train"]) == 1005,     f"train harus 1005 baris (v4), dapat {len(ds['train'])}"
assert len(ds["validation"]) == 102, f"validation harus 102 baris (v4), dapat {len(ds['validation'])}"
assert len(ds["test"]) == 100,       f"test harus 100 baris, dapat {len(ds['test'])}"


### Util: render per model, sanity check, dan eval harness in-notebook

- `render_ds(tk)` — render `messages`+`tools` → kolom `text` memakai chat
  template tokenizer yang diberikan (cell konversi dari `finetune/README.md`). `arguments` (JSON string) di-parse jadi dict
  SEBELUM `apply_chat_template` — template diam-diam membuang argumen non-dict.
- `sanity_check(dsm)` — WAJIB lolos sebelum training: round-trip render→parse
  vs gold (format tool-call JSON Qwen3) + cek format v4 (reminder di ujung
  system block, history berupa penanda).
- `run_eval(tag, m, tk)` — replay split `test`, definisi metrik **persis**
  `finetune/eval_tool_calling.py`. Kondisi meniru produksi: **thinking NYALA**
  (terverifikasi via `/api/chat` bahwa Ollama default thinking-on), sampling
  produksi `temperature 0.7 / top_p 0.8` (`app/core/config.py`), cap 768 token
  (= `num_predict` harness; termasuk token thinking — thinking terpotong =
  "tanpa call", sama seperti di jalur produksi). Diagnostik `truncated_rows` /
  `think_unclosed_rows` / `avg_gen_tokens` memisahkan "salah pilih tool" dari
  "kehabisan jatah token".

In [ ]:
import json
import random
import re
import time
import torch
from collections import defaultdict
from tqdm.auto import tqdm

def render_ds(tk):
    """Render dataset mentah → kolom `text` dengan chat template model ybs."""
    def to_text(ex):
        msgs = ex["messages"]
        for m in msgs:
            for tc in (m.get("tool_calls") or []):
                if isinstance(tc["function"]["arguments"], str):
                    tc["function"]["arguments"] = json.loads(tc["function"]["arguments"])
        return {"text": tk.apply_chat_template(
            msgs, tools = json.loads(ex["tools_json"]),
            tokenize = False, add_generation_prompt = False)}
    return ds.map(to_text)

# ── v4: paritas serving untuk EVAL ────────────────────────────────────────────
# Ollama meng-collate semua system message ke blok system TERATAS (join "\n\n");
# baris train/val v4 sudah tersimpan dalam bentuk itu (system diakhiri
# TOOL_REMINDER, history sudah penanda _history_view). Split TEST dibekukan
# dalam format lama (v1) -> saat replay dirakit ulang persis runtime:
# system TERKINI (+ blok FAQ baris itu) + history penanda + reminder + tanya.
FAQ_HEADER = "\n\nKONTEKS FAQ (jawab pertanyaan umum berdasarkan ini):\n"
_sys_full = next(m[0]["content"] for m in ds["train"]["messages"]
                 if FAQ_HEADER not in m[0]["content"])
CUR_SYSTEM, _sep, _rem = _sys_full.rpartition("\n\nINGAT ATURAN TOOL")
TOOL_REMINDER = "INGAT ATURAN TOOL" + _rem
assert CUR_SYSTEM.startswith("Kamu adalah asisten") and "WAJIB" in TOOL_REMINDER

def history_view(c):
    """Replika _history_view produksi (app/llm/agent.py)."""
    if c.startswith("Berikut menu"):
        return "[Aku sudah menampilkan daftar menu via tool get_menu]"
    if c.startswith("*") and "Harga:" in c:
        produk = c.split("*")[1] if c.count("*") >= 2 else "produk"
        return f"[Aku sudah menampilkan detail {produk} + fotonya via tool get_product_detail]"
    if len(c) > 200:
        return c[:200] + " …(dipotong)"
    return c

def runtime_msgs(row):
    """row['messages'][:-1] (format lama maupun v4) -> rakitan paritas runtime."""
    msgs = [dict(x) for x in row["messages"][:-1]]
    sysc = msgs[0]["content"]
    faq = sysc[sysc.index(FAQ_HEADER):] if FAQ_HEADER in sysc else ""
    faq = faq.split("\n\nINGAT ATURAN TOOL")[0]
    out = [{"role": "system", "content": CUR_SYSTEM + faq + "\n\n" + TOOL_REMINDER}]
    for m in msgs[1:]:
        if m["role"] == "user":
            out.append(m)
        else:
            c = m.get("content") or ""
            out.append({"role": "assistant",
                        "content": c if c.endswith("…(dipotong)") else history_view(c)})
    return out

def strip_nones(o):
    if isinstance(o, dict):
        return {k: strip_nones(v) for k, v in o.items() if v is not None}
    if isinstance(o, list):
        return [strip_nones(v) for v in o]
    return o

def gold_of(row):
    """Tool emas (nama, args) dari turn assistant final — (None, None) utk baris non-tool."""
    final = row["messages"][-1]
    if final.get("tool_calls"):
        fn = final["tool_calls"][0]["function"]
        raw = fn["arguments"]
        return fn["name"], (json.loads(raw) if isinstance(raw, str) else strip_nones(raw))
    return None, None

def canon_args(obj):
    """Normalisasi utk exact-match — persis finetune/eval_tool_calling.py."""
    if isinstance(obj, dict):
        return {k: canon_args(v) for k, v in sorted(obj.items())}
    if isinstance(obj, list):
        return [canon_args(v) for v in obj]
    if isinstance(obj, str) and obj.isdigit():
        return obj  # string angka tetap string; qty salah tipe dihitung salah
    return obj

def parse_first_tool_call(text):
    """(nama, args, invalid) dari output mentah Qwen3:
    <tool_call>{"name": "nama", "arguments": {...}}</tool_call>.
    Blok thinking dibuang dulu; terpotong tanpa </think> = tidak sempat menjawab."""
    text = text.split("<|im_end|>")[0]
    if "</think>" in text:
        text = text.split("</think>", 1)[1]
    elif "<think>" in text:
        text = ""
    if "<tool_call>" not in text:
        return None, None, False
    body = text.split("<tool_call>", 1)[1].split("</tool_call>")[0].strip()
    try:
        obj = json.loads(body)
        return obj["name"], (obj.get("arguments") or {}), False
    except Exception:
        return None, None, True  # mulai <tool_call> tapi tidak bisa diparse

def sanity_check(dsm):
    """WAJIB sebelum training: render→parse round-trip harus cocok dengan gold."""
    random.seed(42)
    tool_idx  = [i for i, m in enumerate(dsm["train"]["messages"]) if m[-1].get("tool_calls")]
    plain_idx = [i for i, m in enumerate(dsm["train"]["messages"]) if not m[-1].get("tool_calls")]
    for i in random.sample(tool_idx, 25):
        row = dsm["train"][i]
        gname, gargs = gold_of(row)
        out = row["text"].rsplit("<|im_start|>assistant\n", 1)[1]
        name, args, invalid = parse_first_tool_call(out + "<|im_end|>")
        assert name == gname and not invalid, f"baris {i}: {name!r} != {gname!r} (invalid={invalid})"
        assert canon_args(args) == canon_args(gargs), f"baris {i}: args {args} != {gargs}"
    for i in random.sample(plain_idx, 10):
        out = dsm["train"][i]["text"].rsplit("<|im_start|>assistant\n", 1)[1]
        name, _a, inv = parse_first_tool_call(out + "<|im_end|>")
        assert name is None and not inv, f"baris {i}: false tool di baris non-tool"
    # v4: reminder di UJUNG system block (bukan system message tengah) dan
    # history assistant harus penanda/pendek — persis yang dilihat Ollama.
    for i in random.sample(range(len(dsm["train"])), 50):
        row = dsm["train"][i]
        assert row["messages"][0]["content"].endswith(TOOL_REMINDER), f"baris {i}: reminder hilang"
        assert row["text"].count("<|im_start|>system") == 1, f"baris {i}: system message ganda"
        for m in row["messages"][1:-1]:
            if m["role"] == "assistant" and isinstance(m.get("content"), str):
                assert "Berikut menu" not in m["content"] and "Harga:" not in m["content"], \
                    f"baris {i}: history literal lolos penanda"
    print(f"sanity render OK ({len(tool_idx)} baris tool / {len(plain_idx)} non-tool)")
    print("===== CONTOH AKHIR RENDER BARIS TOOL-CALL =====")
    print(dsm["train"][tool_idx[0]]["text"][-500:])

def run_eval(tag, m, tk, batch_size = 16, max_new_tokens = 768, chat_kwargs = None):
    """Replay seluruh split test; definisi metrik persis eval_tool_calling.py.
    max_new_tokens 768 = cap num_predict harness. THINKING default NYALA
    (enable_thinking=True) — paritas dgn default Ollama di produksi/harness."""
    from unsloth import FastLanguageModel
    if chat_kwargs is None:
        chat_kwargs = {"enable_thinking": True}
    FastLanguageModel.for_inference(m)
    for _mod in m.modules():  # buang cache rope_deltas basi (lihat catatan training)
        if hasattr(_mod, "rope_deltas"):
            _mod.rope_deltas = None
    torch.manual_seed(42)  # sampling produksi, seed tetap → antar-run sebanding
    t0 = time.time()

    prompts, golds, rtypes = [], [], []
    for row in ds["test"]:
        msgs = runtime_msgs(row)  # v4: rakitan paritas runtime (penanda + reminder)
        prompts.append(tk.apply_chat_template(
            msgs, tools = json.loads(row["tools_json"]),
            tokenize = False, add_generation_prompt = True, **chat_kwargs))
        golds.append(gold_of(row))
        rtypes.append(row["meta"]["type"])

    tok = tk
    old_side, tok.padding_side = tok.padding_side, "left"
    outputs = []
    for b in tqdm(range(0, len(prompts), batch_size), desc = tag):
        enc = tok(prompts[b:b + batch_size], return_tensors = "pt",
                  padding = True).to("cuda")
        # sampling = config PRODUKSI chatbot (app/core/config.py: temperature 0.7,
        # top_p 0.8 — juga dipakai harness lokal), bukan default Modelfile
        out = m.generate(**enc, max_new_tokens = max_new_tokens, do_sample = True,
                         temperature = 0.7, top_p = 0.8, top_k = 20,
                         use_cache = True, pad_token_id = tok.pad_token_id)
        outputs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:])
    tok.padding_side = old_side

    per_type = defaultdict(lambda: {"n": 0, "sel": 0, "param": 0, "irrelevant_ok": 0,
                                    "false_tool": 0, "invalid": 0})
    for (gname, gargs), rtype, raw in zip(golds, rtypes, outputs):
        name, args, invalid = parse_first_tool_call(raw)
        st = per_type[rtype]
        st["n"] += 1
        if gname is None:            # baris non-tool
            if name is not None:
                st["false_tool"] += 1
            else:                    # percobaan invalid tanpa call ikut sini — mirror harness
                st["irrelevant_ok"] += 1
        else:                        # baris tool
            if invalid and name is None:
                st["invalid"] += 1
            if name == gname:
                st["sel"] += 1
                if canon_args(args) == canon_args(gargs):
                    st["param"] += 1

    tool_types = [t for t in per_type if t.startswith("T")]
    non_types  = [t for t in per_type if t.startswith("N")]
    tool_n = sum(per_type[t]["n"] for t in tool_types)
    non_n  = sum(per_type[t]["n"] for t in non_types)
    agg = {
        "tag": tag,
        "function_selection_acc": round(sum(per_type[t]["sel"] for t in tool_types) / max(1, tool_n), 3),
        "param_exact_acc":        round(sum(per_type[t]["param"] for t in tool_types) / max(1, tool_n), 3),
        "invalid_call_rate":      round(sum(per_type[t]["invalid"] for t in tool_types) / max(1, tool_n), 3),
        "irrelevance_acc":        round(sum(per_type[t]["irrelevant_ok"] for t in non_types) / max(1, non_n), 3),
        "false_tool_rate":        round(sum(per_type[t]["false_tool"] for t in non_types) / max(1, non_n), 3),
        "per_type": {t: dict(per_type[t]) for t in sorted(per_type)},
        "seconds": round(time.time() - t0, 1),
        # Diagnostik: memisahkan "salah pilih tool" dari "kehabisan jatah token".
        "truncated_rows":      sum(1 for o in outputs if "<|im_end|>" not in o),
        "think_unclosed_rows": sum(1 for o in outputs
                                   if "</think>" not in o.split("<|im_end|>")[0]),
        "avg_gen_tokens":      round(sum(len(tok(o).input_ids) for o in outputs)
                                     / max(1, len(outputs)), 1),
    }
    print(json.dumps({k: v for k, v in agg.items() if k != "per_type"}, indent = 2))
    print(f"⏱️ {agg['seconds']}s untuk {len(prompts)} baris "
          f"({agg['seconds'] / max(1, len(prompts)):.1f}s/baris) | "
          f"terpotong cap: {agg['truncated_rows']} | thinking tak selesai: "
          f"{agg['think_unclosed_rows']} | rata-rata {agg['avg_gen_tokens']} token/baris")
    return agg

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

<a name="Pipeline"></a>
### Pipeline: load → sanity → baseline → LoRA train → eval → export

Konfigurasi (sesuai INSTRUKSI):
LoRA r=16, α=16, dropout=0, semua proyeksi linear, seed 42; lr 2e-4 linear,
warmup 10 step, 2 epoch, batch efektif 8, eval val-loss tiap 25 step;
**aturan berhenti**: `EarlyStoppingCallback(patience=4)` + `load_best_model_at_end`
(val-loss tak membaik 2 evaluasi berturut-turut → stop, pakai checkpoint terbaik).

Catatan teknis yang sudah tertanam (jangan diubah):
- Reset `model.rope_deltas` sebelum `trainer.train()` — cache sisa `generate()`
  baseline membuat forward training crash di transformers==5.2.0
  ("size of tensor a (2) must match ... (0)"; diperbaiki di >= 5.5).
- Masking: `train_on_responses_only` marker Qwen LALU final-turn-only (v4) —
  loss hanya pada respons assistant TERAKHIR; diverifikasi dengan decode label.
- Setelah selesai: hanya LoRA → `lora_<tag>/` (cepat), lalu model dibongkar
  dari VRAM. **Export GGUF dipisah** ke seksi akhir.

In [ ]:
import gc
import glob
import os
import matplotlib.pyplot as plt
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# v4: smoke = 4 insiden live WA + kasus lama (dibaca mata, bukan verdict)
SMOKE = [
    ("sapaan — JANGAN panggil tool", [], "halo kak, selamat siang!"),
    ("insiden #1 — menu WAJIB get_menu, bukan hafalan", [], "ada menu apa saja?"),
    ("insiden #2 — detail by name -> get_product_detail", [], "bento cookies kayak gimana ya?"),
    ("insiden #3 — history penanda, TETAP panggil tool",
     [{"role": "user", "content": "menu dong"},
      {"role": "assistant", "content": "[Aku sudah menampilkan daftar menu via tool get_menu]"}],
     "ada menu apa aja?"),
    ("insiden #4 — argumen verbatim 'cupcake'", [], "beli 4 cupcake"),
    ("ambigu N5 — harus bertanya balik", [], "mau pesan kue dong"),
    ("adversarial N6 — BUKAN cancel_order", [], "cara batalin pesanan gimana sih kak?"),
    ("out-of-scope — tolak dengan sopan", [], "kak tau resep rendang yang enak nggak?"),
]

def run_smoke(m, tk, tag):
    """Cek kualitatif singkat (dibaca mata, bukan verdict) — thinking nyala."""
    sys_msg = {"role": "system", "content": CUR_SYSTEM + "\n\n" + TOOL_REMINDER}
    smoke_tools = json.loads(ds["test"][0]["tools_json"])
    tok = tk
    for label, hist, q in SMOKE:
        prompt = tk.apply_chat_template(
            [sys_msg, *hist, {"role": "user", "content": q}], tools = smoke_tools,
            tokenize = False, add_generation_prompt = True, enable_thinking = True)
        enc = tok(prompt, return_tensors = "pt").to("cuda")
        t0 = time.time()
        out = m.generate(**enc, max_new_tokens = 512, do_sample = True,
                         temperature = 0.7, top_p = 0.8, top_k = 20,
                         use_cache = True, pad_token_id = tok.pad_token_id)
        dt = time.time() - t0
        raw = tok.decode(out[0, enc["input_ids"].shape[1]:]).split("<|im_end|>")[0]
        think, _, ans = raw.rpartition("</think>")
        print(f"\n===== [{tag}] {label}  (⏱️ {dt:.1f}s)\nUSER : {q}")
        if think.strip():
            print(f"THINK: {think.strip()[:200]}")
        print(f"BOT  : {ans.strip()[:400]}")

def finetune_and_eval(hf_name, tag):
    print(f"\n{'#' * 70}\n# {tag}  ←  {hf_name}\n{'#' * 70}")

    # ---- 1. Load (jalur teks; max_seq_length 4096 — prompt terpanjang ±1.9k tok)
    model, tokenizer = FastLanguageModel.from_pretrained(
        hf_name,
        max_seq_length = 4096,
        load_in_4bit = False,
        use_gradient_checkpointing = "unsloth",
    )
    tok = tokenizer

    # ---- 2. Render dataset dgn template model INI + sanity (WAJIB lolos)
    dsm = render_ds(tokenizer)
    sanity_check(dsm)

    # ---- 3. LoRA (INSTRUKSI §2)
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
        use_rslora = False,
        loftq_config = None,
    )

    def lora_b_signature():
        """Σ|lora_B| — nol saat init; harus > 0 setelah training yang benar."""
        return sum(p.detach().float().abs().sum().item()
                   for n, p in model.named_parameters() if "lora_B" in n)

    print(f"Σ|lora_B| sebelum training: {lora_b_signature():.6f} (harus 0 — init)")

    # ---- 4. Baseline SEBELUM training (LoRA baru = identitas, aman)
    baseline = run_eval(f"BASELINE {tag}", model, tokenizer)

    # ---- 5. Trainer + masking respons-only
    FastLanguageModel.for_training(model)
    trainer = SFTTrainer(
        model = model,
        processing_class = tok,   # BUKAN Processor multimodal (lihat markdown)
        train_dataset = dsm["train"],
        eval_dataset = dsm["validation"],
        # patience 4 (bukan 2): aturan INSTRUKSI = berhenti saat val-loss NAIK
        # berkelanjutan; EarlyStoppingCallback menghitung "tidak membaik" yang
        # lebih galak — patience 2 menyetop di step 75 saat kurva cuma datar,
        # menyisakan checkpoint ¼-epoch yang kurang latih. load_best_model_at_end
        # tetap jadi pengaman overfit yang sebenarnya.
        callbacks = [EarlyStoppingCallback(early_stopping_patience = 4)],
        args = SFTConfig(
            dataset_text_field = "text",
            max_length = 4096,
            # 2×4 (bukan 8×1): batch 8 bikin EVAL step OOM di T4 (materialisasi
            # logit fp32 batch penuh ~7GB) dan tak lebih cepat (compute-bound).
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            num_train_epochs = 2,
            warmup_steps = 10,
            learning_rate = 2e-4,
            lr_scheduler_type = "linear",
            logging_steps = 5,
            optim = "adamw_8bit",
            weight_decay = 0.001,
            seed = 42,
            output_dir = f"outputs_{tag}",
            report_to = "none",
            per_device_eval_batch_size = 4,  # kecil: eval-loss tak terpengaruh, hindari OOM logit
            eval_strategy = "steps",
            eval_steps = 25,
            save_strategy = "steps",
            save_steps = 25,
            save_total_limit = 2,
            load_best_model_at_end = True,
            metric_for_best_model = "eval_loss",
            greater_is_better = False,
        ),
    )
    ct = getattr(tok, "chat_template", None) or getattr(tokenizer, "chat_template", "")
    assert "<|im_start|>user" in ct and "<|im_start|>assistant" in ct, (
        "Marker Qwen tidak ditemukan di chat_template aktual!")
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part    = "<|im_start|>assistant\n",
    )

    # v4 (akar insiden #1): loss HANYA pada respons assistant FINAL.
    # train_on_responses_only membuka SEMUA turn assistant — termasuk history
    # (ringkasan cart dsb) -> model v3 menghafal teks itu lalu menirunya di
    # produksi. Di serving, model memang hanya pernah menghasilkan SATU turn.
    _asst_ids = tok("<|im_start|>assistant\n", add_special_tokens = False).input_ids

    def _final_only(ex):
        ids, labels = ex["input_ids"], list(ex["labels"])
        last, n = -1, len(_asst_ids)
        for i in range(len(ids) - n + 1):
            if ids[i] == _asst_ids[0] and ids[i:i + n] == _asst_ids:
                last = i
        if last > 0:
            labels[:last] = [-100] * last
        return {"labels": labels}

    trainer.train_dataset = trainer.train_dataset.map(_final_only)
    trainer.eval_dataset = trainer.eval_dataset.map(_final_only)

    # verifikasi masking: token ber-loss harus HANYA respons assistant FINAL
    mt_idx = next(i for i in range(len(dsm["train"]))
                  if len(dsm["train"][i]["messages"]) > 3)
    for _idx in (0, mt_idx):
        ex = trainer.train_dataset[_idx]
        labels = ex["labels"]
        kept = [t for t, l in zip(ex["input_ids"], labels) if l != -100]
        assert 0 < len(kept) < len(labels), "masking gagal total"
        decoded_kept = tok.decode(kept)
        assert "Toti Cakery, sebuah toko kue" not in decoded_kept, (
            "System prompt ikut terlatih — masking SALAH, stop!")
        assert "Ringkasan pesananmu" not in decoded_kept, (
            "history assistant ikut terlatih — masking final-only GAGAL")
        assert "[Aku sudah menampilkan" not in decoded_kept, (
            "penanda history ikut terlatih — masking final-only GAGAL")
    print(f"masking OK (final-turn-only): baris {mt_idx} (multi-turn) "
          f"{len(kept)}/{len(labels)} token dilatih: {decoded_kept[:120]!r}")

    # ---- 6. Train (reset rope_deltas dulu — cache sisa generate() baseline)
    for _m in model.modules():
        if hasattr(_m, "rope_deltas"):
            _m.rope_deltas = None
    trainer_stats = trainer.train()
    print(f"training {tag}: {trainer_stats.metrics['train_runtime']:.0f}s "
          f"({trainer_stats.metrics['train_runtime'] / 60:.1f} menit)")

    lora_b_after = lora_b_signature()
    print(f"Σ|lora_B| setelah training: {lora_b_after:.6f}")
    if lora_b_after == 0.0:
        print("🚨 LoRA TIDAK BELAJAR SAMA SEKALI (lora_B masih nol) — hasil "
              "fine-tuned di bawah = base model. Jangan pakai GGUF-nya; "
              "laporkan ke Kevin/Claude sebelum lanjut.")

    # ---- 7. Kurva loss (cek overfitting: val naik saat train turun = jelek)
    hist = trainer.state.log_history
    tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
    ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    for s, l in ev:
        print(f"  step {s:>4}  eval_loss {l:.4f}")
    fig, ax = plt.subplots(figsize = (7, 3))
    if tr: ax.plot(*zip(*tr), color = "#4477AA", lw = 2, label = "train loss")
    if ev: ax.plot(*zip(*ev), color = "#EE7733", lw = 2, marker = "o", ms = 6, label = "val loss")
    ax.set_xlabel("step"); ax.set_ylabel("loss")
    ax.grid(alpha = 0.25); ax.legend(frameon = False)
    ax.set_title(f"Train vs validation loss — {tag}", loc = "left")
    plt.tight_layout(); plt.show()

    # ---- 8. Eval SESUDAH training (kondisi identik dgn baseline) + smoke
    finetuned = run_eval(f"FINE-TUNED {tag}", model, tokenizer)
    run_smoke(model, tokenizer, tag)

    # ---- 9. Simpan LoRA saja (cepat, ±50MB). Export GGUF sengaja DIPISAH ke
    # seksi setelah tabel perbandingan — prioritas: hasil semua model dulu,
    # export hanya untuk model yang dipilih.
    model.save_pretrained(f"lora_{tag}")
    tokenizer.save_pretrained(f"lora_{tag}")
    print(f"LoRA tersimpan: lora_{tag}/ — export GGUF di seksi akhir notebook")

    # ---- 10. Bongkar dari VRAM utk model berikutnya
    del trainer, model, tokenizer, tok, dsm
    gc.collect()
    torch.cuda.empty_cache()
    return {"baseline": baseline, "finetuned": finetuned,
            "train_seconds": round(trainer_stats.metrics["train_runtime"], 1)}

### Jalankan training (resumable)

Kalau Colab timeout di tengah: restart runtime → jalankan ulang cell instalasi,
data, util, pipeline → lalu cell ini lagi. Error CUDA/VRAM aneh → restart
runtime (paling bersih).

In [ ]:
all_results = globals().get("all_results", {})
all_results["toti-qwen3-17b"] = finetune_and_eval("unsloth/Qwen3-1.7B", "toti-qwen3-17b")

<a name="Compare"></a>
### Tabel perbandingan akhir — baseline → fine-tuned vs target

Kolom `harness` = model produksi v3 diukur di harness v4 lokal
(`finetune/results_toti-qwen-1.7b.harness-v4.json`) sebagai jangkar realitas.
Gerbang rilis: fine-tuned >= jangkar itu di semua metrik, DAN skenario regresi
R1-R6 lolos saat diverifikasi harness lokal setelah deploy.

In [ ]:
TARGETS = {
    "function_selection_acc": (">=", 0.80),
    "param_exact_acc":        (">=", 0.65),
    "irrelevance_acc":        (">=", 0.80),
    "false_tool_rate":        ("<=", 0.15),
    "invalid_call_rate":      ("<=", 0.05),  # "tetap ~0"
}
# Jangkar pembanding (kolom `harness`) = model PRODUKSI saat ini (toti-qwen-1.7b
# v3) diukur di HARNESS v4 lokal, 16 Jul 2026
# (finetune/results_toti-qwen-1.7b.harness-v4.json). Gerbang rilis v4: metrik
# >= jangkar ini DAN skenario regresi R1-R6 lolos (scenario_suite.py, lokal —
# v3 gagal R1/R3/R6: menjawab menu dari hafalan, dan R5-args: mengarang varian).
HARNESS_BASELINE = {
    "toti-qwen3-17b": {"function_selection_acc": 0.900, "param_exact_acc": 0.783,
                       "irrelevance_acc": 0.900, "false_tool_rate": 0.100,
                       "invalid_call_rate": 0.000},
}

scoreboard = {}
for tag, res in all_results.items():
    b, f = res["baseline"], res["finetuned"]
    h = HARNESS_BASELINE.get(tag)
    print(f"\n===== {tag} =====")
    print(f"{'metrik':<24} {'harness':>8} {'base-nb':>8} {'ft-nb':>8} {'delta':>7}  target   status")
    hits = 0
    for mname, (op, tgt) in TARGETS.items():
        hv = f"{h[mname]:.3f}" if h else "-"
        hit = (f[mname] >= tgt) if op == ">=" else (f[mname] <= tgt)
        hits += hit
        print(f"{mname:<24} {hv:>8} {b[mname]:>8.3f} {f[mname]:>8.3f} "
              f"{f[mname] - b[mname]:>+7.3f}  {op} {tgt:<4}  {'✅' if hit else '❌'}")
    scoreboard[tag] = hits
    print(f"target tercapai: {hits}/{len(TARGETS)} | waktu eval ft: {f['seconds']}s | "
          f"trunc: {f['truncated_rows']} | training: {res['train_seconds']}s")
    print("per-type (baseline → fine-tuned):")
    for t in sorted(f["per_type"]):
        bb, ff = b["per_type"].get(t, {}), f["per_type"][t]
        if t.startswith("T"):
            print(f"  {t:4} n={ff['n']:3}  sel {bb.get('sel', 0):3} → {ff['sel']:3}   "
                  f"param {bb.get('param', 0):3} → {ff['param']:3}")
        else:
            print(f"  {t:4} n={ff['n']:3}  ok  {bb.get('irrelevant_ok', 0):3} → {ff['irrelevant_ok']:3}   "
                  f"false_tool {bb.get('false_tool', 0):3} → {ff['false_tool']:3}")

# Arsip kuantitatif utk tugas akhir: baseline (sebelum) vs fine-tuned (sesudah),
# per metrik agregat dan per tipe skenario, kondisi eval identik (seed, sampling,
# prompt paritas produksi). Unduh file ini sebagai lampiran bab evaluasi.
with open("hasil_metrik_v4.json", "w") as fjson:
    json.dump(all_results, fjson, indent = 2)
print("\n📊 hasil_metrik_v4.json tersimpan (baseline vs fine-tuned per metrik/tipe)")

if scoreboard:
    best = max(scoreboard, key = lambda t: (scoreboard[t],
               all_results[t]["finetuned"]["function_selection_acc"]))
    print(f"\n🏆 Kandidat terbaik in-notebook: {best} "
          f"({scoreboard[best]}/{len(TARGETS)} target) — KONFIRMASI dengan harness "
          f"lokal setelah deploy GGUF (verdict resmi).")

<a name="Export"></a>
### Export GGUF — jalankan SETELAH tabel perbandingan

LoRA sudah di disk (`lora_toti-qwen3-17b/`); cell ini me-reload lalu
mengonversi ke GGUF **Q4_K_M** (±10-15 menit). Konversi Unsloth membangun
sumber f16 dulu lalu meng-quantize — sesuai aturan §6 PROMPT_FINETUNE_V4
(Q4 harus dari f16, bukan re-quantize Q8).

In [ ]:
# Cell ini MANDIRI: aman dijalankan di session baru (setelah restart) selama
# folder lora_<tag>/ masih ada di disk — cukup jalankan cell instalasi dulu.
import gc
import glob
import os
import torch
from unsloth import FastLanguageModel

# Default: semua LoRA yang ditemukan di disk (bukan dari variabel memori)
EXPORT_TAGS = [d[len("lora_"):] for d in sorted(glob.glob("lora_*")) if os.path.isdir(d)]
print("akan di-export:", EXPORT_TAGS or "(tidak ada lora_*/ di disk!)")

for tag in EXPORT_TAGS:
    print(f"\n===== export {tag} =====")
    m2, t2 = FastLanguageModel.from_pretrained(
        f"lora_{tag}",
        max_seq_length = 4096,
        load_in_4bit = False,
    )
    try:
        m2.save_pretrained_gguf(f"gguf_{tag}", t2, quantization_method = "q4_k_m")
        for f in glob.glob(f"gguf_{tag}*/**/*.gguf", recursive = True) + glob.glob(f"gguf_{tag}*.gguf"):
            print("GGUF:", f, f"{os.path.getsize(f) / 1e9:.2f} GB")
        # Alternatif download manual (±1GB): push langsung ke HF
        # m2.push_to_hub_gguf(f"HF_USERNAME/{tag}-gguf", t2, token = "HF_TOKEN")
    except Exception as exc:
        print(f"⚠️ Export GGUF gagal {tag}: {exc} — LoRA tetap aman di lora_{tag}/")
    del m2, t2
    gc.collect()
    torch.cuda.empty_cache()

### Push GGUF ke HuggingFace — jalankan SETELAH cell export

Meng-upload `.gguf` yang baru dibuat ke repo model HF (default **privat**).
Nama file di Hub sudah = nama produksi yang dipakai Modelfile di `finetune/`
(`toti-qwen-1.7b.Q4_K_M.gguf.v4`) → di mesin lokal
tinggal `huggingface-cli download` lalu `ollama create`, tanpa rename.

Token: dipakai token login HF yang sama seperti saat load model (`get_token()`);
kalau kosong, cell akan meminta lewat prompt. Butuh token **write**.


In [ ]:
# =========================================================================
# Push GGUF -> HuggingFace Hub (jalankan SETELAH cell export GGUF di atas)
# =========================================================================
import glob, os, getpass
from huggingface_hub import HfApi, create_repo, get_token

HF_REPO    = "LasagnaS/toti-qwen-gguf"   # repo model tujuan
HF_PRIVATE = True                        # repo privat (ubah ke False bila publik)

# Nama file akhir di Hub per tag = persis nama yang diharapkan Modelfile lokal.
HUB_NAME = {
    "toti-qwen3-17b": "toti-qwen-1.7b.Q4_K_M.gguf.v4",
}

token = os.environ.get("HF_TOKEN") or get_token() or getpass.getpass("HF token (write): ")
api = HfApi(token=token)
create_repo(HF_REPO, repo_type="model", private=HF_PRIVATE, exist_ok=True, token=token)

pushed = []
for tag, dst in HUB_NAME.items():
    hits = sorted(glob.glob(f"gguf_{tag}*/**/*.gguf", recursive=True) +
                  glob.glob(f"gguf_{tag}*.gguf"))
    if not hits:
        print(f"skip {tag}: tak ada .gguf (export dulu cell di atas)")
        continue
    src = hits[0]
    print(f"upload {tag}: {src} -> {HF_REPO}/{dst} ({os.path.getsize(src)/1e9:.2f} GB) ...")
    api.upload_file(path_or_fileobj=src, path_in_repo=dst,
                    repo_id=HF_REPO, repo_type="model")
    pushed.append(dst)

print("\nselesai. ter-push:", pushed or "(tidak ada)")
print(f"  https://huggingface.co/{HF_REPO}/tree/main")
# Download di mesin lokal Kevin:
#   hf download LasagnaS/toti-qwen-gguf toti-qwen-1.7b.Q4_K_M.gguf.v4 --local-dir finetune/


### Deploy ke Ollama + verdict resmi (di mesin lokal Kevin)

Download GGUF (atau via HF), taruh di `finetune/`, lalu:

| Tag | Rename GGUF menjadi | Perintah create |
|---|---|---|
| `toti-qwen3-17b` | `toti-qwen-1.7b.Q4_K_M.gguf.v4` | `ollama create toti-qwen-1.7b-v4 -f Modelfile.qwen3-1.7b-v4` |

`Modelfile.qwen3-1.7b-v4` SUDAH ada di `finetune/` dengan Go-TEMPLATE asli
`qwen3:1.7b`. **Template beda = penyebab #1 "kok jadi bodoh setelah
export"** — jangan tulis Modelfile sendiri.

```bash
cd /home/kevin/clcode/chatbot
# verdict resmi per model (±50 mnt-2.5 jam per model di CPU; jalankan background):
chatbot-service/.venv/bin/python finetune/eval_tool_calling.py --model toti-qwen-1.7b-v4
chatbot-service/.venv/bin/python finetune/scenario_suite.py     --model toti-qwen-1.7b-v4
# Gerbang rilis v4 (§5): bandingkan dgn baseline v3 pada HARNESS YANG SAMA
# (results_toti-qwen-1.7b.harness-v4.json — sudah dijalankan sebelum training)
# + SEMUA skenario regresi R1-R6 harus LOLOS (regression_pass di scenario_*.json)
```

Model pemenang → `LLM_MODEL=<nama>` di `.env`, smoke `python -m scripts.chat_cli`,
dan `pytest chatbot-service/tests/` harus tetap hijau. Gagal target → diagnosis
INSTRUKSI §4 (sanity render → masking → 3 epoch / lr 1e-4 → per-type).